# PhishGuard: 03. Baseline Machine Learning Models & Evaluation

## Research & Security Context
> **Objective:** Train, evaluate, and compare baseline ML models using strictly the 27 static URL-derived features from `extract_url_features(url)`.

In cybersecurity, **Recall** and **False Negatives (FN)** are paramount. A False Positive results in a momentary nuisance (a user is warned about a benign site), but a **False Negative allows an active phishing attack to compromise credentials or infrastructure**.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import get_train_test_data
from src.model import get_model_pipeline, train_pipeline, evaluate_pipeline
from src.feature_extractor import FEATURE_NAMES

sns.set_theme(style="whitegrid")
print(f"Setup complete. Total URL features utilized: {len(FEATURE_NAMES)}")

## 1. Load Stratified Train-Test Splits
To prevent data leakage, preprocessing transforms (such as `StandardScaler`) are fit strictly inside pipelines on the training set only.

In [ ]:
X_train, X_test, y_train, y_test = get_train_test_data(test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Testing set:  {X_test.shape[0]:,} samples")
print(f"Features (27 URL-only):\n{list(X_train.columns)}")

## 2. Train Baseline Classifiers
We train:
1. **Logistic Regression** (Linear baseline with `StandardScaler`)
2. **Decision Tree** (Interpretable non-linear decision boundary)
3. **Random Forest** (Non-linear ensemble with variance reduction)

In [ ]:
models = {
    "LogisticRegression": get_model_pipeline("logistic_regression"),
    "DecisionTree": get_model_pipeline("decision_tree"),
    "RandomForest": get_model_pipeline("random_forest")
}

eval_results = {}
for name, pipe in models.items():
    print(f"Training {name}...")
    train_pipeline(pipe, X_train, y_train)
    metrics = evaluate_pipeline(pipe, X_test, y_test)
    eval_results[name] = metrics
    print(f"  -> Accuracy: {metrics['accuracy']:.4f} | Recall: {metrics['recall']:.4f} | F1: {metrics['f1_score']:.4f} | FN: {metrics['confusion_matrix']['false_negatives']}")

## 3. Comparative Evaluation Table

In [ ]:
summary_rows = []
for name, m in eval_results.items():
    cm = m["confusion_matrix"]
    summary_rows.append({
        "Model": name,
        "Accuracy": m["accuracy"],
        "Precision": m["precision"],
        "Recall (Phishing)": m["recall"],
        "F1-Score": m["f1_score"],
        "ROC-AUC": m["roc_auc"],
        "False Negatives (Missed Phish)": cm["false_negatives"],
        "False Positives (False Alarms)": cm["false_positives"]
    })

df_summary = pd.DataFrame(summary_rows)
df_summary

## 4. Confusion Matrices Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), dpi=150)
for ax, (name, m) in zip(axes, eval_results.items()):
    cm = m["confusion_matrix"]
    matrix = np.array([
        [cm["true_negatives"], cm["false_positives"]],
        [cm["false_negatives"], cm["true_positives"]]
    ])
    sns.heatmap(matrix, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Legit (0)", "Phish (1)"],
                yticklabels=["Legit (0)", "Phish (1)"])
    ax.set_title(f"{name}\nMissed (FN): {cm['false_negatives']}", fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## 5. Feature Importances (Random Forest)
We inspect which static URL characteristics contribute most significantly to detecting malicious links.

In [ ]:
rf_clf = models["RandomForest"].named_steps["classifier"]
feat_imp = pd.DataFrame({
    "Feature": FEATURE_NAMES,
    "Importance": rf_clf.feature_importances_
}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(9, 6), dpi=150)
sns.barplot(data=feat_imp.head(12), x="Importance", y="Feature", palette="viridis")
plt.title("Top 12 Most Influential URL Features (Random Forest)", fontweight="bold")
plt.xlabel("Gini Importance")
plt.show()
feat_imp.head(10)